# Módulo 3 · Clase 3
## Cuando una recta no alcanza: árboles y Random Forest

[Abrir este notebook en Google Colab](https://colab.research.google.com/github/cmosquerat/arca-diplomado/blob/agent/modulo3-clase3-codex/clase3-codex/Clase3_No_Linealidad_Arboles_RandomForest.ipynb)

**Dos problemas:** estimar producción (regresión) y reconocer arenisca (clasificación). El material se carga directamente desde GitHub.

---
# 0 · Preparación

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score, accuracy_score, recall_score, confusion_matrix, ConfusionMatrixDisplay


---
# 1 · Ver la no linealidad

Una recta obliga a que el efecto sea constante. Lunas y círculos muestran fronteras que una logística no puede doblar.

In [ ]:
from sklearn.datasets import make_moons, make_circles
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

X_moon, y_moon = make_moons(n_samples=500, noise=.16, random_state=12)
X_circle, y_circle = make_circles(n_samples=500, noise=.10, factor=.42, random_state=12)


In [ ]:
fig, axs = plt.subplots(1,2,figsize=(10,4))
for ax, X, y, title in [(axs[0],X_moon,y_moon,'Lunas'),(axs[1],X_circle,y_circle,'Círculos')]:
    ax.scatter(X[:,0],X[:,1],c=y,cmap='coolwarm',s=16)
    ax.set_title(title)
plt.show()


In [ ]:
for nombre, modelo in [('Logística', LogisticRegression()),('Árbol', DecisionTreeClassifier(max_depth=5,random_state=42))]:
    modelo.fit(X_moon,y_moon)
    print(nombre, 'accuracy lunas:', round(modelo.score(X_moon,y_moon),3))


---
# 2 · Cómo aprende un árbol

El árbol prueba variables y cortes. Escoge la pregunta que deja grupos más homogéneos y repite el proceso en cada rama.

In [ ]:
from sklearn.tree import plot_tree
arbol_demo = DecisionTreeClassifier(max_depth=2, random_state=42).fit(X_moon,y_moon)
plt.figure(figsize=(14,6))
plot_tree(arbol_demo,filled=True,rounded=True,impurity=False,proportion=True)
plt.show()


## Actividad corta

Lee una ruta completa: comienza en la raíz, elige verdadero/falso y termina en una hoja. ¿Qué clase predice y con qué proporción?

---
# 3 · Problema de regresión: medidor virtual de flujo

In [ ]:
URL_VOLVE = 'https://raw.githubusercontent.com/cmosquerat/arca-diplomado/agent/modulo3-clase3-codex/clase3-codex/operacion_pozos_volve.csv'
volve = pd.read_csv(URL_VOLVE)
print(volve.shape)
volve.head()


## Diccionario de variables

|Variable|Significado|
|---|---|
|`horas`|horas en operación ese día|
|`p_fondo`|presión de fondo|
|`p_cabeza`|presión de cabeza|
|`t_cabeza`|temperatura en cabeza|
|`choke`|apertura del choke|
|`dp_choke`|caída de presión en el choke|
|`oil`|producción medida; objetivo|

In [ ]:
features = ['horas','p_fondo','p_cabeza','t_cabeza','choke','dp_choke']
X, y = volve[features], volve['oil']
Xtr, Xte, ytr, yte = train_test_split(X,y,test_size=.25,random_state=42)


In [ ]:
from sklearn.linear_model import LinearRegression
lineal=LinearRegression().fit(Xtr,ytr)
p=lineal.predict(Xte)
print('Lineal R²:',round(r2_score(yte,p),3),'MAE:',round(mean_absolute_error(yte,p)))


In [ ]:
from sklearn.tree import DecisionTreeRegressor
arbol=DecisionTreeRegressor(max_depth=5,random_state=42).fit(Xtr,ytr)
p=arbol.predict(Xte)
print('Árbol R²:',round(r2_score(yte,p),3),'MAE:',round(mean_absolute_error(yte,p)))


## Profundidad y sobreajuste

Prueba 2, 5, 10 y sin límite. Un desempeño perfecto en entrenamiento puede ser una señal de memoria.

In [ ]:
filas=[]
for depth in [2,5,10,None]:
    m=DecisionTreeRegressor(max_depth=depth,random_state=42).fit(Xtr,ytr)
    filas.append([depth,m.score(Xtr,ytr),m.score(Xte,yte),mean_absolute_error(yte,m.predict(Xte))])
pd.DataFrame(filas,columns=['profundidad','R² train','R² test','MAE test']).round(3)


---
# 4 · Random Forest para regresión

In [ ]:
from sklearn.ensemble import RandomForestRegressor
bosque=RandomForestRegressor(n_estimators=200,random_state=42,n_jobs=-1).fit(Xtr,ytr)
p_rf=bosque.predict(Xte)
print('Bosque R²:',round(r2_score(yte,p_rf),3),'MAE:',round(mean_absolute_error(yte,p_rf)))


In [ ]:
pd.Series(bosque.feature_importances_,index=features).sort_values().plot.barh(title='Importancia de variables')
plt.show()


---
# 5 · Segundo dataset: clasificación de litología FORCE 2020

Volvemos al problema de la Clase 2: arenisca o lutita. Ahora permitimos que el modelo aprenda fronteras no lineales.

In [ ]:
URL_FORCE = 'https://raw.githubusercontent.com/cmosquerat/arca-diplomado/agent/modulo3-clase3-codex/clase3-codex/litologia_force2020.csv'
lito = pd.read_csv(URL_FORCE)
lito['y']=(lito['LITH']=='Sandstone').astype(int)
lito[['GR','RHOB','NPHI','DTC','RDEP','LITH']].head()


In [ ]:
features_lito=['GR','RHOB','NPHI','DTC','RDEP']
X=lito[features_lito]; y=lito['y']
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.25,random_state=42,stratify=y)


In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
modelos={
 'Logística':make_pipeline(StandardScaler(),LogisticRegression(max_iter=1000)),
 'Árbol':DecisionTreeClassifier(max_depth=8,random_state=42),
 'Bosque':RandomForestClassifier(n_estimators=200,random_state=42,n_jobs=-1)}
resultados=[]
for nombre,m in modelos.items():
    m.fit(Xtr,ytr); pred=m.predict(Xte)
    tn,fp,fn,tp=confusion_matrix(yte,pred).ravel()
    resultados.append([nombre,accuracy_score(yte,pred),recall_score(yte,pred),fn*10+fp])
pd.DataFrame(resultados,columns=['modelo','accuracy','recall arena','costo']).round(3)


## Actividad final

1. ¿Qué modelo reduce más el costo de la Clase 2?
2. Cambia `max_depth` del árbol.
3. Explica por qué el bosque puede ganar precisión y a la vez perder explicabilidad.

---
# Cierre

- Un árbol aprende preguntas y cortes, no una fórmula curva.
- En regresión, una hoja predice un promedio.
- En clasificación, una hoja vota y produce probabilidades.
- La profundidad controla la tensión entre aprender y memorizar.
- Random Forest estabiliza muchos árboles distintos.